# WaveGuard – CSI Data Exploration

This notebook walks through the key stages of the WaveGuard ML pipeline:

1. **Data loading** – reading CSI windows from an HDF5 file  
2. **Preprocessing** – Hampel filter → bandpass → phase unwrap → PCA  
3. **Visualisation** – amplitude heatmaps, frequency spectra, class distribution  
4. **Model forward pass** – running WiFlexFormer and CSIClassifier on a dummy batch

Set `DATA_PATH` below to point at your WaveGuard HDF5 file, or use the
synthetic-data helper at the bottom to run everything without real data.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

# Add the repo root to the path so we can import from ml/
REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

%matplotlib inline
plt.rcParams["figure.dpi"] = 110
plt.rcParams["font.size"] = 11

print("Python:", sys.version)
print("PyTorch:", torch.__version__)

## 1. Synthetic data (no HDF5 file needed)

We generate a small batch of synthetic CSI windows – 50 samples × 52 subcarriers × 500
time steps – with per-class signal patterns so that the visualisations look realistic.

In [ ]:
N_SAMPLES    = 50
N_CHANNELS   = 52   # subcarriers
N_TIME       = 500  # samples at 100 Hz → 5-second window
FS           = 100.0
CLASS_NAMES  = ["empty", "presence", "movement", "fall"]
N_CLASSES    = len(CLASS_NAMES)

rng = np.random.default_rng(0)

def make_synthetic_csi(label: int, rng: np.random.Generator) -> np.ndarray:
    """Return a (channels, time) float32 CSI window for the given label."""
    t = np.linspace(0, N_TIME / FS, N_TIME)
    base = rng.standard_normal((N_CHANNELS, N_TIME)).astype(np.float32) * 0.1

    if label == 0:  # empty – low-amplitude noise
        return base
    if label == 1:  # presence – slow DC-like variation
        signal = 0.5 * np.sin(2 * np.pi * 0.3 * t)
        return base + signal[np.newaxis, :]
    if label == 2:  # movement – mid-frequency burst
        signal = np.sin(2 * np.pi * 1.5 * t) * np.exp(-0.5 * ((t - 2.5) / 0.8) ** 2)
        return base + signal[np.newaxis, :]
    # label == 3 – fall – sharp transient
    signal = np.exp(-((t - 2.0) / 0.15) ** 2)
    return base + signal[np.newaxis, :]


labels = rng.integers(0, N_CLASSES, size=N_SAMPLES)
data   = np.stack([make_synthetic_csi(int(l), rng) for l in labels])  # (N, C, T)

print(f"Synthetic dataset: {data.shape}, dtype={data.dtype}")
print("Label counts:", {CLASS_NAMES[i]: int((labels == i).sum()) for i in range(N_CLASSES)})

## 2. Class distribution

In [ ]:
counts = [int((labels == i).sum()) for i in range(N_CLASSES)]

fig, ax = plt.subplots(figsize=(5, 3))
bars = ax.bar(CLASS_NAMES, counts, color=["#4C9BE8", "#56C98A", "#F5A623", "#E84C4C"])
ax.bar_label(bars, padding=3)
ax.set_ylabel("Samples")
ax.set_title("Class distribution (synthetic)")
plt.tight_layout()
plt.show()

## 3. Raw CSI amplitude heatmaps

Each row is a subcarrier; columns are time steps.  Warmer colours indicate
higher amplitude.

In [ ]:
fig, axes = plt.subplots(1, N_CLASSES, figsize=(14, 3), sharey=True)

for cls_idx, ax in enumerate(axes):
    # Pick first sample of this class
    sample_idx = int(np.where(labels == cls_idx)[0][0])
    csi_window = data[sample_idx]  # (52, 500)
    im = ax.imshow(
        csi_window,
        aspect="auto",
        origin="lower",
        cmap="viridis",
        extent=[0, N_TIME / FS, 0, N_CHANNELS],
    )
    ax.set_title(CLASS_NAMES[cls_idx])
    ax.set_xlabel("Time (s)")

axes[0].set_ylabel("Subcarrier index")
fig.colorbar(im, ax=axes, label="Amplitude")
fig.suptitle("Raw CSI amplitude – one sample per class", y=1.01)
plt.tight_layout()
plt.show()

## 4. Preprocessing pipeline

We run a single window through the `CSIPipeline` and compare the raw vs.
preprocessed amplitude on a single subcarrier.

In [ ]:
from ml.preprocessing.csi_pipeline import CSIPipeline

# Build pipeline (amplitude-only, 52 channels)
pipeline = CSIPipeline(
    n_subcarriers=N_CHANNELS,
    hampel_window=5,
    hampel_sigma=3.0,
    bp_low_hz=0.1,
    bp_high_hz=20.0,
    fs=FS,
    pca_components=3,
)

# Choose the first 'movement' sample
mov_idx = int(np.where(labels == 2)[0][0])
raw     = data[mov_idx]                           # (52, 500) float32
proc    = pipeline(raw.astype(np.float64)).astype(np.float32)

t_axis = np.linspace(0, N_TIME / FS, N_TIME)
subcarrier = 10  # pick a representative subcarrier

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axes[0].plot(t_axis, raw[subcarrier], lw=0.8, color="steelblue")
axes[0].set_title(f"Raw CSI amplitude – subcarrier {subcarrier} (movement)")
axes[0].set_ylabel("Amplitude")

axes[1].plot(t_axis, proc[subcarrier], lw=0.8, color="coral")
axes[1].set_title("After preprocessing (Hampel → BPF → PCA)")
axes[1].set_ylabel("Amplitude")
axes[1].set_xlabel("Time (s)")

plt.tight_layout()
plt.show()

## 5. Power spectral density

Compare the PSD before and after bandpass filtering for the movement sample.

In [ ]:
from scipy.signal import welch

freqs_raw,  psd_raw  = welch(raw[subcarrier],  fs=FS, nperseg=256)
freqs_proc, psd_proc = welch(proc[subcarrier], fs=FS, nperseg=256)

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(freqs_raw,  psd_raw,  lw=1.2, label="Raw",           color="steelblue")
ax.semilogy(freqs_proc, psd_proc, lw=1.2, label="Preprocessed",  color="coral")
ax.axvline(0.1, ls="--", color="gray", lw=0.8, label="BPF 0.1 Hz")
ax.axvline(20,  ls="--", color="gray", lw=0.8, label="BPF 20 Hz")
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("PSD")
ax.set_title("Power spectral density – movement sample")
ax.legend()
ax.set_xlim(0, FS / 2)
plt.tight_layout()
plt.show()

## 6. WiFlexFormer – model architecture

We instantiate the default WiFlexFormer and print a parameter summary.

In [ ]:
from ml.models.wiflexformer import WiFlexFormer

model = WiFlexFormer(
    in_channels=N_CHANNELS,
    embed_dim=32,
    num_heads=16,
    num_layers=4,
    num_classes=N_CLASSES,
)
print(model)
print(f"\nTotal trainable parameters: {model.num_parameters:,}")

## 7. Model forward pass

Run a batch of preprocessed samples through WiFlexFormer and CSIClassifier.

In [ ]:
from ml.models.csi_classifier import CSIClassifier

# Preprocess the full synthetic dataset
processed = np.stack([
    pipeline(data[i].astype(np.float64)).astype(np.float32)
    for i in range(N_SAMPLES)
])  # (N, 52, T)

# Normalise each sample
mean = processed.mean(axis=-1, keepdims=True)
std  = processed.std(axis=-1, keepdims=True) + 1e-8
processed = (processed - mean) / std

x_tensor = torch.from_numpy(processed)   # (N, 52, T)
y_tensor = torch.from_numpy(labels.astype(np.int64))

model.eval()
with torch.no_grad():
    logits_wf = model(x_tensor)

print("WiFlexFormer logits shape:", logits_wf.shape)
print("WiFlexFormer predictions: ", logits_wf.argmax(dim=-1).numpy())

lite_model = CSIClassifier(in_channels=N_CHANNELS, num_classes=N_CLASSES)
lite_model.eval()
with torch.no_grad():
    logits_csi = lite_model(x_tensor)

print("CSIClassifier logits shape:", logits_csi.shape)
print(f"CSIClassifier params: {lite_model.num_parameters:,}")

## 8. Gaussian positional encoding visualisation

The WiFlexFormer uses Gaussian basis functions (σ=10) rather than
sinusoidal encoding.  Here we visualise the encoding for a 64-step
sequence with embed_dim=32.

In [ ]:
from ml.models.wiflexformer import GaussianPositionalEncoding

pe_module = GaussianPositionalEncoding(embed_dim=32, max_len=200, sigma=10.0)
pe_matrix = pe_module.pe.squeeze(0).numpy()  # (200, 32)

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(pe_matrix[:100].T, aspect="auto", cmap="RdBu_r", origin="lower")
ax.set_xlabel("Position (time step)")
ax.set_ylabel("Encoding dimension")
ax.set_title("Gaussian Positional Encoding (σ=10, embed_dim=32, first 100 positions)")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## 9. Loading real data (optional)

Replace `DATA_PATH` with the path to your WaveGuard HDF5 file.  The
expected schema is:

```
/csi      – float32 (N, channels, time)
/labels   – int64   (N,)
```

In [ ]:
DATA_PATH = "/data/waveguard/waveguard.h5"  # ← change me

if Path(DATA_PATH).exists():
    from ml.preprocessing.data_loader import WaveGuardDataset

    train_ds = WaveGuardDataset(path=DATA_PATH, split="train")
    val_ds   = WaveGuardDataset(path=DATA_PATH, split="val")
    test_ds  = WaveGuardDataset(path=DATA_PATH, split="test")

    print(f"Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}")

    x0, y0 = train_ds[0]
    print(f"Sample shape: {x0.shape}, label: {CLASS_NAMES[y0.item()]}")
else:
    print(f"Dataset not found at {DATA_PATH!r} – skipping real-data section.")